# 04. Behavioral calibration

Calibrates the reduced-form promotion lift, displacement strength, and persistence distribution used by the policy model.

## 1. Imports and configuration

In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.optimize import minimize_scalar

warnings.filterwarnings("ignore", category=FutureWarning)

CURRENT_DIR = Path.cwd().resolve()
POSSIBLE_ROOTS = [CURRENT_DIR, CURRENT_DIR.parent]

PROJECT_ROOT = next(
    (
        root
        for root in POSSIBLE_ROOTS
        if (root / "data" / "processed").is_dir()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        f"Could not locate the project root from {CURRENT_DIR}"
    )

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
TABLE_DIR = PROJECT_ROOT / "results" / "final" / "tables"
FIGURE_DIR = PROJECT_ROOT / "results" / "final" / "figures"
MODEL_DIR = PROJECT_ROOT / "results" / "models"
DEMAND_ARTIFACT_DIR = PROJECT_ROOT / "artifacts" / "demand"
CALIBRATION_ARTIFACT_DIR = PROJECT_ROOT / "artifacts" / "calibration"

for directory in [
    PROCESSED_DIR,
    TABLE_DIR,
    FIGURE_DIR,
    MODEL_DIR,
    DEMAND_ARTIFACT_DIR,
    CALIBRATION_ARTIFACT_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

DATA_PATH = (
    PROCESSED_DIR
    / "cereal_demand_model_data.parquet"
)
PREDICTION_PATH = (
    DEMAND_ARTIFACT_DIR
    / "demand_predictions.pkl"
)

PRIMARY_MODEL = "product_promotion"

# Event-study design.
EVENT_WINDOW = np.arange(-4, 7, dtype=int)
PRE_EVENT_WEEKS = (-3, -2, -1)
POST_LAGS_FOR_CALIBRATION = (1, 2, 3, 4)
INITIAL_PRE_PROMO_GAP = 3
INITIAL_POST_PROMO_GAP = 3
MIN_ISOLATED_EVENTS = 40

# Promotions with almost no measured price discount are retained in
# the descriptive event study, but excluded from the depth-based
# behavioral calibration.
MIN_CALIBRATION_DISCOUNT = 0.05
FALLBACK_CALIBRATION_DISCOUNT = 0.02
MIN_DEPTH_EVENTS = 80

# Bootstrap.
RANDOM_SEED = 42
BOOTSTRAP_REPLICATIONS = 200
EVENT_STUDY_CI = (0.05, 0.95)

# Behavioral parameter bounds.
ELASTICITY_BOUNDS = (0.10, 5.00)
PROMOTION_LIFT_BOUNDS = (-1.00, 3.00)
DISPLACEMENT_BOUNDS = (0.00, 3.00)
PERSISTENCE_BOUNDS = (0.05, 0.95)
MIN_PERSISTENCE_SIGNAL = 0.05
MAX_PERSISTENCE_SLOPE_RATIO = 1.25

# Used only when post-period slopes do not identify persistence.
PERSISTENCE_PARTIAL_ID_GRID = (0.15, 0.35, 0.60)

GROSS_MARGIN_COLUMN = "gross_margin_pct_observed"
MIN_VALID_MARGIN = -1.0
MAX_VALID_MARGIN = 0.95

PARAMETER_OUTPUT_PATH = (
    CALIBRATION_ARTIFACT_DIR
    / "pooled_behavioral_draws.pkl"
)
BASE_PARAMETER_OUTPUT_PATH = (
    CALIBRATION_ARTIFACT_DIR
    / "pooled_behavioral_bootstrap.pkl"
)
EVENT_OUTPUT_PATH = (
    CALIBRATION_ARTIFACT_DIR
    / "isolated_promotion_events.pkl"
)
EVENT_STUDY_TABLE_PATH = (
    TABLE_DIR
    / "promotion_event_study.csv"
)
DEPTH_EVENT_STUDY_TABLE_PATH = (
    TABLE_DIR
    / "promotion_event_study_by_depth.csv"
)
MOMENT_TABLE_PATH = (
    TABLE_DIR
    / "pooled_calibration_moments.csv"
)
PARAMETER_SUMMARY_PATH = (
    TABLE_DIR
    / "pooled_behavioral_parameter_summary.csv"
)
CONFIGURATION_PATH = (
    MODEL_DIR
    / "behavioral_calibration.json"
)
EVENT_STUDY_FIGURE_PATH = (
    FIGURE_DIR
    / "promotion_event_study.png"
)
DEPTH_EVENT_STUDY_FIGURE_PATH = (
    FIGURE_DIR
    / "promotion_event_study_by_depth.png"
)

print("Project root:", PROJECT_ROOT)
print("Python:", sys.version.split()[0])
print("Data:", DATA_PATH)
print("Predictions:", PREDICTION_PATH)

## 2. Load the selected empirical panels

In [ ]:
if not DATA_PATH.is_file():
    raise FileNotFoundError(f"Demand data not found: {DATA_PATH}")

history_raw = pd.read_parquet(DATA_PATH)

if PREDICTION_PATH.is_file():
    predictions = pd.read_pickle(PREDICTION_PATH)
    required_prediction_columns = {
        "store_upc",
        "week",
        "model",
        "split",
    }
    missing = required_prediction_columns.difference(
        predictions.columns
    )
    if missing:
        raise ValueError(
            "Prediction artifact is missing columns: "
            f"{sorted(missing)}"
        )

    predictions["store_upc"] = (
        predictions["store_upc"].astype(str)
    )
    predictions["week"] = pd.to_numeric(
        predictions["week"],
        errors="coerce",
    )

    primary_predictions = predictions.loc[
        predictions["model"].astype(str).eq(PRIMARY_MODEL)
    ].copy()

    if primary_predictions.empty:
        raise ValueError(
            f"No rows for model {PRIMARY_MODEL!r}."
        )

    selected_panels = set(
        primary_predictions["store_upc"].dropna().unique()
    )

    calibration_weeks = primary_predictions.loc[
        primary_predictions["split"].astype(str).eq("calibration"),
        "week",
    ].dropna()

    if calibration_weeks.empty:
        raise ValueError(
            "The rolling prediction artifact has no calibration weeks."
        )

    CALIBRATION_END_WEEK = int(calibration_weeks.max())
    evaluation_weeks = primary_predictions.loc[
        primary_predictions["split"].astype(str).eq("test"),
        "week",
    ].dropna()
    EVALUATION_START_WEEK = (
        int(evaluation_weeks.min())
        if not evaluation_weeks.empty
        else CALIBRATION_END_WEEK + 1
    )
else:
    panel_counts = (
        history_raw.assign(
            store_upc=history_raw["store_upc"].astype(str)
        )
        .groupby("store_upc", observed=True)
        .size()
        .sort_values(ascending=False)
    )
    selected_panels = set(panel_counts.head(120).index)
    week_values = pd.to_numeric(
        history_raw["week"],
        errors="coerce",
    ).dropna()
    CALIBRATION_END_WEEK = int(
        np.quantile(week_values, 0.80)
    )
    EVALUATION_START_WEEK = CALIBRATION_END_WEEK + 1
    warnings.warn(
        "The rolling prediction artifact was not found. "
        "The notebook selected the 120 longest panels and used "
        "an 80% chronological split."
    )

print("Selected panels:", len(selected_panels))
print("Calibration end week:", CALIBRATION_END_WEEK)
print("Evaluation start week:", EVALUATION_START_WEEK)

## 3. Clean prices, promotions, demand, and unit costs

In [ ]:
def first_existing_column(
    frame: pd.DataFrame,
    candidates: list[str],
    required: bool = True,
) -> str | None:
    for column in candidates:
        if column in frame.columns:
            return column
    if required:
        raise KeyError(
            "None of the candidate columns were found: "
            f"{candidates}"
        )
    return None


def numeric_series(
    frame: pd.DataFrame,
    candidates: list[str],
    default: float = np.nan,
) -> pd.Series:
    column = first_existing_column(
        frame,
        candidates,
        required=False,
    )
    if column is None:
        return pd.Series(default, index=frame.index, dtype=float)
    return pd.to_numeric(frame[column], errors="coerce")


history = history_raw.copy()

for column in ["store", "upc", "store_upc"]:
    history[column] = history[column].astype(str)

history = history.loc[
    history["store_upc"].isin(selected_panels)
].copy()

history["week"] = pd.to_numeric(
    history["week"],
    errors="coerce",
)
history["move"] = pd.to_numeric(
    history["move"],
    errors="coerce",
)

price_column = first_existing_column(
    history,
    [
        "model_unit_price",
        "unit_price_observed",
        "unit_price",
        "price",
    ],
)
history["price"] = pd.to_numeric(
    history[price_column],
    errors="coerce",
)

regular_price_column = first_existing_column(
    history,
    ["regular_price", "reference_price"],
    required=False,
)
if regular_price_column is None:
    history["regular_price_model"] = (
        history.groupby("store_upc", observed=True)["price"]
        .transform(
            lambda values: values.rolling(
                13,
                min_periods=4,
            ).max()
        )
    )
else:
    history["regular_price_model"] = pd.to_numeric(
        history[regular_price_column],
        errors="coerce",
    )

history["regular_price_model"] = (
    history["regular_price_model"]
    .where(history["regular_price_model"].gt(0))
    .fillna(history["price"])
)

discount_column = first_existing_column(
    history,
    ["discount_depth_model", "discount_depth"],
    required=False,
)
if discount_column is None:
    history["discount_depth_model"] = (
        1.0
        - history["price"]
        / history["regular_price_model"]
    )
else:
    history["discount_depth_model"] = pd.to_numeric(
        history[discount_column],
        errors="coerce",
    )

history["discount_depth_model"] = (
    history["discount_depth_model"]
    .fillna(0.0)
    .clip(0.0, 0.80)
)

promotion_column = first_existing_column(
    history,
    [
        "promotion_indicator",
        "promo_recorded",
        "promo_from_discount",
    ],
    required=False,
)

if promotion_column is None:
    state_column = first_existing_column(
        history,
        ["pricing_state", "promo_state"],
        required=False,
    )
    if state_column is None:
        promotion_indicator = (
            history["discount_depth_model"].ge(0.03)
        )
    else:
        state_text = (
            history[state_column]
            .astype("string")
            .fillna("")
            .str.lower()
        )
        promotion_indicator = (
            state_text.str.contains("promo", na=False)
            & ~state_text.str.contains("post", na=False)
        )
else:
    promotion_indicator = (
        pd.to_numeric(
            history[promotion_column],
            errors="coerce",
        )
        .fillna(0)
        .gt(0)
    )

history["promotion_indicator"] = (
    promotion_indicator
    | history["discount_depth_model"].ge(0.03)
).astype(int)

post_column = first_existing_column(
    history,
    ["post_promotion_indicator", "post_promo"],
    required=False,
)
if post_column is None:
    history["post_promotion_indicator"] = (
        history.groupby("store_upc", observed=True)[
            "promotion_indicator"
        ]
        .shift(1)
        .fillna(0)
        .astype(int)
    )
else:
    history["post_promotion_indicator"] = (
        pd.to_numeric(
            history[post_column],
            errors="coerce",
        )
        .fillna(0)
        .gt(0)
        .astype(int)
    )

price_imputed = (
    history["price_imputed"].fillna(False).astype(bool)
    if "price_imputed" in history.columns
    else pd.Series(False, index=history.index)
)
sales_outlier = (
    history["sales_outlier"].fillna(False).astype(bool)
    if "sales_outlier" in history.columns
    else pd.Series(False, index=history.index)
)

valid = (
    history["week"].notna()
    & history["move"].notna()
    & np.isfinite(history["move"])
    & history["move"].ge(0)
    & history["price"].notna()
    & np.isfinite(history["price"])
    & history["price"].gt(0)
    & ~price_imputed
    & ~sales_outlier
)
history = history.loc[valid].copy()
history["week"] = history["week"].astype(int)

if history.duplicated(["store_upc", "week"]).any():
    raise ValueError(
        "Duplicate store-product-week observations were found."
    )

history["log1p_move"] = np.log1p(history["move"])
history["log_price"] = np.log(history["price"])

if GROSS_MARGIN_COLUMN not in history.columns:
    raise KeyError(
        f"Required margin column {GROSS_MARGIN_COLUMN!r} "
        "was not found."
    )

raw_margin = pd.to_numeric(
    history[GROSS_MARGIN_COLUMN],
    errors="coerce",
)
finite_margin = raw_margin.loc[
    raw_margin.notna() & np.isfinite(raw_margin)
]
if finite_margin.empty:
    raise ValueError("No usable gross-margin observations.")

margin_scale = (
    100.0
    if finite_margin.abs().quantile(0.95) > 1.5
    else 1.0
)
history["gross_margin_fraction"] = raw_margin / margin_scale
valid_margin = history["gross_margin_fraction"].between(
    MIN_VALID_MARGIN,
    MAX_VALID_MARGIN,
    inclusive="both",
)
history["unit_cost_observed"] = (
    history["price"]
    * (1.0 - history["gross_margin_fraction"])
)
history.loc[
    ~valid_margin
    | ~np.isfinite(history["unit_cost_observed"])
    | history["unit_cost_observed"].le(0),
    "unit_cost_observed",
] = np.nan

history["sample_period"] = np.where(
    history["week"].le(CALIBRATION_END_WEEK),
    "calibration",
    "evaluation",
)

history = history.sort_values(
    ["store_upc", "week"]
).reset_index(drop=True)

print("Clean rows:", f"{len(history):,}")
print("Panels:", history["store_upc"].nunique())
print(
    "Promotion share:",
    f"{history['promotion_indicator'].mean():.2%}",
)
print(
    "Usable unit-cost share:",
    f"{history['unit_cost_observed'].notna().mean():.2%}",
)
print(
    "Gross-margin scale:",
    "percentage" if margin_scale == 100.0 else "fraction",
)

## 4. Remove panel, product-week, and store-week demand components

In [ ]:
def alternating_multiway_residual(
    frame: pd.DataFrame,
    value_column: str,
    group_columns: list[str],
    max_iterations: int = 300,
    tolerance: float = 1e-10,
) -> pd.Series:
    values = pd.to_numeric(
        frame[value_column],
        errors="coerce",
    ).astype(float)

    if values.isna().any():
        raise ValueError(
            f"{value_column!r} contains missing values."
        )

    residual = values - values.mean()

    for _ in range(max_iterations):
        previous = residual.to_numpy(copy=True)

        for group_column in group_columns:
            residual = (
                residual
                - residual.groupby(
                    frame[group_column],
                    observed=True,
                ).transform("mean")
            )

        change = float(
            np.max(
                np.abs(
                    residual.to_numpy()
                    - previous
                )
            )
        )
        if change < tolerance:
            break

    return residual


calibration_history = history.loc[
    history["sample_period"].eq("calibration")
].copy()

calibration_history["upc_week"] = (
    calibration_history["upc"].astype(str)
    + "_"
    + calibration_history["week"].astype(str)
)
calibration_history["store_week"] = (
    calibration_history["store"].astype(str)
    + "_"
    + calibration_history["week"].astype(str)
)

residual_groups = [
    "store_upc",
    "upc_week",
    "store_week",
]

calibration_history[
    "demand_multiway_residual"
] = alternating_multiway_residual(
    calibration_history,
    "log1p_move",
    residual_groups,
)
calibration_history[
    "price_multiway_residual"
] = alternating_multiway_residual(
    calibration_history,
    "log_price",
    residual_groups,
)

print(
    "Residual demand mean:",
    f"{calibration_history['demand_multiway_residual'].mean():.3e}",
)
print(
    "Residual price mean:",
    f"{calibration_history['price_multiway_residual'].mean():.3e}",
)
print("Absorbed effects:", residual_groups)

## 5. Identify isolated one-week promotions

In [ ]:
def isolated_promotion_mask(
    frame: pd.DataFrame,
    pre_gap: int,
    post_gap: int,
) -> pd.Series:
    grouped = frame.groupby(
        "store_upc",
        observed=True,
        sort=False,
    )
    mask = frame["promotion_indicator"].eq(1)

    for offset in range(1, pre_gap + 1):
        previous_promo = grouped[
            "promotion_indicator"
        ].shift(offset)
        previous_week = grouped[
            "week"
        ].shift(offset)

        mask &= (
            previous_promo.eq(0)
            & frame["week"]
            .sub(previous_week)
            .eq(offset)
        )

    for offset in range(1, post_gap + 1):
        next_promo = grouped[
            "promotion_indicator"
        ].shift(-offset)
        next_week = grouped[
            "week"
        ].shift(-offset)

        mask &= (
            next_promo.eq(0)
            & next_week
            .sub(frame["week"])
            .eq(offset)
        )

    return mask.fillna(False)


selected_gap = None
isolated_mask = None

for pre_gap, post_gap in [
    (3, 3),
    (2, 2),
    (1, 1),
]:
    candidate_mask = isolated_promotion_mask(
        calibration_history,
        pre_gap=pre_gap,
        post_gap=post_gap,
    )
    event_count = int(candidate_mask.sum())

    print(
        f"Isolated events with {pre_gap}/{post_gap} gaps:",
        event_count,
    )

    selected_gap = (pre_gap, post_gap)
    isolated_mask = candidate_mask

    if event_count >= MIN_ISOLATED_EVENTS:
        break

events = calibration_history.loc[
    isolated_mask,
    [
        "store",
        "upc",
        "store_upc",
        "week",
        "discount_depth_model",
        "price",
        "regular_price_model",
    ],
].copy()

events = events.rename(
    columns={
        "week": "event_week",
        "discount_depth_model": (
            "event_discount_depth"
        ),
        "price": "event_price",
        "regular_price_model": (
            "event_regular_price"
        ),
    }
)
events["event_id"] = np.arange(
    len(events),
    dtype=int,
)

if len(events) < 10:
    raise RuntimeError(
        "Too few isolated promotions for a "
        "meaningful prototype calibration."
    )

selected_calibration_discount = (
    MIN_CALIBRATION_DISCOUNT
)

depth_event_count = int(
    events["event_discount_depth"]
    .ge(selected_calibration_discount)
    .sum()
)

if depth_event_count < MIN_DEPTH_EVENTS:
    selected_calibration_discount = (
        FALLBACK_CALIBRATION_DISCOUNT
    )
    depth_event_count = int(
        events["event_discount_depth"]
        .ge(selected_calibration_discount)
        .sum()
    )
    warnings.warn(
        "Fewer than the requested number of promotions "
        "had at least a 5% measured discount. "
        "The depth calibration threshold was lowered "
        "to 2%."
    )

print("Selected gap rule:", selected_gap)
print("Isolated promotion events:", len(events))
print(
    "Events used for depth calibration:",
    depth_event_count,
)
print(
    "Minimum calibration discount:",
    selected_calibration_discount,
)
print(
    "Panels with events:",
    events["store_upc"].nunique(),
)

## 6. Build the promotion event study

In [ ]:
event_blocks = []

event_source = calibration_history[
    [
        "store_upc",
        "week",
        "demand_multiway_residual",
        "price_multiway_residual",
        "move",
        "promotion_indicator",
        "discount_depth_model",
    ]
].copy()

for relative_week in EVENT_WINDOW:
    block = events[
        [
            "event_id",
            "store_upc",
            "event_week",
            "event_discount_depth",
        ]
    ].copy()

    block["relative_week"] = int(
        relative_week
    )
    block["week"] = (
        block["event_week"]
        + int(relative_week)
    )

    block = block.merge(
        event_source,
        on=[
            "store_upc",
            "week",
        ],
        how="left",
        validate="many_to_one",
    )
    event_blocks.append(block)

event_long = pd.concat(
    event_blocks,
    ignore_index=True,
)

pre_means = (
    event_long.loc[
        event_long[
            "relative_week"
        ].isin(PRE_EVENT_WEEKS)
    ]
    .groupby(
        "event_id",
        observed=True,
    )
    .agg(
        pre_demand_residual_mean=(
            "demand_multiway_residual",
            "mean",
        ),
        pre_price_residual_mean=(
            "price_multiway_residual",
            "mean",
        ),
        pre_demand_observations=(
            "demand_multiway_residual",
            "count",
        ),
        pre_price_observations=(
            "price_multiway_residual",
            "count",
        ),
    )
    .reset_index()
)

event_long = event_long.merge(
    pre_means,
    on="event_id",
    how="left",
    validate="many_to_one",
)

event_long = event_long.loc[
    event_long[
        "pre_demand_observations"
    ].ge(2)
    & event_long[
        "pre_price_observations"
    ].ge(2)
].copy()

event_long["event_effect"] = (
    event_long[
        "demand_multiway_residual"
    ]
    - event_long[
        "pre_demand_residual_mean"
    ]
)

event_long[
    "event_price_change_residual"
] = (
    event_long[
        "price_multiway_residual"
    ]
    - event_long[
        "pre_price_residual_mean"
    ]
)

event_summary = (
    event_long.pivot_table(
        index=[
            "event_id",
            "store_upc",
            "event_discount_depth",
        ],
        columns="relative_week",
        values="event_effect",
        aggfunc="first",
    )
    .reset_index()
)

current_price_change = (
    event_long.loc[
        event_long[
            "relative_week"
        ].eq(0),
        [
            "event_id",
            "event_price_change_residual",
        ],
    ]
    .drop_duplicates("event_id")
    .rename(
        columns={
            "event_price_change_residual": (
                "current_price_change_residual"
            )
        }
    )
)

event_summary = event_summary.merge(
    current_price_change,
    on="event_id",
    how="left",
    validate="one_to_one",
)

event_summary = event_summary.dropna(
    subset=[
        0,
        1,
        "current_price_change_residual",
    ]
).copy()

event_summary = event_summary.rename(
    columns={
        0: "current_lift_log",
    }
)

for lag in POST_LAGS_FOR_CALIBRATION:
    effect_column = (
        f"post{lag}_effect_log"
    )
    dip_column = (
        f"post{lag}_dip_log"
    )

    if lag in event_summary.columns:
        event_summary = event_summary.rename(
            columns={
                lag: effect_column,
            }
        )
        event_summary[dip_column] = (
            -event_summary[
                effect_column
            ]
        )
    else:
        event_summary[effect_column] = np.nan
        event_summary[dip_column] = np.nan

event_summary[
    "calibration_event"
] = event_summary[
    "event_discount_depth"
].ge(
    selected_calibration_discount
)

event_summary["depth_group"] = (
    "unpriced_or_very_shallow"
)

qualifying_index = event_summary.index[
    event_summary[
        "calibration_event"
    ]
]

if len(qualifying_index) >= 3:
    depth_codes, depth_bins = pd.qcut(
        event_summary.loc[
            qualifying_index,
            "event_discount_depth",
        ],
        q=3,
        labels=False,
        retbins=True,
        duplicates="drop",
    )

    number_of_bins = (
        len(depth_bins) - 1
    )
    standard_labels = [
        "shallow",
        "medium",
        "deep",
    ]

    if number_of_bins == 3:
        labels = standard_labels
    else:
        labels = [
            f"depth_bin_{index + 1}"
            for index in range(
                number_of_bins
            )
        ]

    event_summary.loc[
        qualifying_index,
        "depth_group",
    ] = [
        labels[int(code)]
        for code in depth_codes
    ]

if event_summary.empty:
    raise RuntimeError(
        "No complete promotion events remained "
        "after constructing the event window."
    )

event_long.to_pickle(
    EVENT_OUTPUT_PATH
)

display_columns = [
    "event_discount_depth",
    "current_price_change_residual",
    "current_lift_log",
] + [
    f"post{lag}_dip_log"
    for lag in POST_LAGS_FOR_CALIBRATION
]

print(
    "Complete events:",
    len(event_summary),
)
print(
    "Median discount depth:",
    f"{event_summary['event_discount_depth'].median():.3f}",
)
print(
    "Raw-depth versus residualized-price-change correlation:",
    f"{event_summary[['event_discount_depth', 'current_price_change_residual']].corr().iloc[0, 1]:.3f}",
)
display(
    event_summary[
        display_columns
    ].describe(
        percentiles=[
            0.10,
            0.50,
            0.90,
        ]
    )
)
display(
    event_summary.groupby(
        "depth_group",
        observed=True,
    )
    .agg(
        events=(
            "event_id",
            "size",
        ),
        median_discount=(
            "event_discount_depth",
            "median",
        ),
        mean_current_lift=(
            "current_lift_log",
            "mean",
        ),
        mean_post1_dip=(
            "post1_dip_log",
            "mean",
        ),
    )
    .reset_index()
)

## 7. Estimate regular-state price elasticity

In [ ]:
regular_rows = calibration_history.loc[
    calibration_history[
        "promotion_indicator"
    ].eq(0)
    & calibration_history[
        "post_promotion_indicator"
    ].eq(0)
].copy()

panel_price_sufficient = (
    regular_rows.assign(
        xx=lambda frame: (
            frame[
                "price_multiway_residual"
            ]
            ** 2
        ),
        xy=lambda frame: (
            frame[
                "price_multiway_residual"
            ]
            * frame[
                "demand_multiway_residual"
            ]
        ),
    )
    .groupby(
        "store_upc",
        observed=True,
    )
    .agg(
        xx=("xx", "sum"),
        xy=("xy", "sum"),
        observations=(
            "week",
            "size",
        ),
    )
    .reset_index()
)

denominator = (
    panel_price_sufficient[
        "xx"
    ].sum()
)

if denominator <= 1e-12:
    raise RuntimeError(
        "Insufficient within-panel price variation "
        "to estimate the calibration elasticity."
    )

regular_log_price_coefficient = float(
    panel_price_sufficient[
        "xy"
    ].sum()
    / denominator
)

absolute_price_elasticity = float(
    np.clip(
        -regular_log_price_coefficient,
        *ELASTICITY_BOUNDS,
    )
)

print(
    "Multiway-FE regular-state "
    "log-price coefficient:",
    f"{regular_log_price_coefficient:.4f}",
)
print(
    "Absolute elasticity used for "
    "point calibration:",
    f"{absolute_price_elasticity:.4f}",
)
print(
    "Interpretation warning: this is a "
    "reduced-form calibration coefficient, "
    "not an identified causal elasticity."
)

## 8. Cluster bootstrap calibration moments

In [ ]:
def weighted_mean(
    values: pd.Series,
    weights: pd.Series,
) -> float:
    valid = (
        values.notna()
        & weights.notna()
        & np.isfinite(values)
        & np.isfinite(weights)
        & weights.gt(0)
    )

    if not valid.any():
        return np.nan

    return float(
        np.average(
            values.loc[valid],
            weights=weights.loc[valid],
        )
    )


def weighted_slope(
    x: pd.Series,
    y: pd.Series,
    weights: pd.Series,
) -> float:
    valid = (
        x.notna()
        & y.notna()
        & weights.notna()
        & np.isfinite(x)
        & np.isfinite(y)
        & np.isfinite(weights)
        & weights.gt(0)
    )

    if valid.sum() < 3:
        return np.nan

    x_values = x.loc[
        valid
    ].to_numpy(dtype=float)
    y_values = y.loc[
        valid
    ].to_numpy(dtype=float)
    weight_values = weights.loc[
        valid
    ].to_numpy(dtype=float)

    x_mean = np.average(
        x_values,
        weights=weight_values,
    )
    y_mean = np.average(
        y_values,
        weights=weight_values,
    )

    denominator = np.sum(
        weight_values
        * (
            x_values - x_mean
        )
        ** 2
    )

    if denominator <= 1e-12:
        return np.nan

    return float(
        np.sum(
            weight_values
            * (
                x_values - x_mean
            )
            * (
                y_values - y_mean
            )
        )
        / denominator
    )


calibration_event_summary = (
    event_summary.loc[
        event_summary[
            "calibration_event"
        ]
    ]
    .copy()
)

event_panels = np.array(
    sorted(
        calibration_event_summary[
            "store_upc"
        ].unique()
    )
)
price_panels = np.array(
    sorted(
        panel_price_sufficient[
            "store_upc"
        ].unique()
    )
)
common_panels = np.intersect1d(
    event_panels,
    price_panels,
)

if len(common_panels) < 5:
    raise RuntimeError(
        "Fewer than five panels contribute to both "
        "event and price calibration."
    )

event_summary_boot = (
    calibration_event_summary.loc[
        calibration_event_summary[
            "store_upc"
        ].isin(common_panels)
    ]
    .copy()
)

price_sufficient_boot = (
    panel_price_sufficient.loc[
        panel_price_sufficient[
            "store_upc"
        ].isin(common_panels)
    ]
    .copy()
)

rng = np.random.default_rng(
    RANDOM_SEED
)
bootstrap_rows = []

for bootstrap_id in range(
    BOOTSTRAP_REPLICATIONS
):
    sampled_panels = rng.choice(
        common_panels,
        size=len(common_panels),
        replace=True,
    )

    counts = (
        pd.Series(sampled_panels)
        .value_counts()
        .rename("cluster_weight")
    )

    events_boot = (
        event_summary_boot.merge(
            counts,
            left_on="store_upc",
            right_index=True,
            how="inner",
            validate="many_to_one",
        )
    )

    price_boot = (
        price_sufficient_boot.merge(
            counts,
            left_on="store_upc",
            right_index=True,
            how="inner",
            validate="one_to_one",
        )
    )

    event_weights = (
        events_boot[
            "cluster_weight"
        ].astype(float)
    )
    price_weights = (
        price_boot[
            "cluster_weight"
        ].astype(float)
    )

    elasticity_denominator = float(
        np.sum(
            price_weights
            * price_boot["xx"]
        )
    )
    elasticity_numerator = float(
        np.sum(
            price_weights
            * price_boot["xy"]
        )
    )

    log_price_coefficient = (
        elasticity_numerator
        / elasticity_denominator
        if elasticity_denominator > 1e-12
        else regular_log_price_coefficient
    )

    elasticity = float(
        np.clip(
            -log_price_coefficient,
            *ELASTICITY_BOUNDS,
        )
    )

    depth = events_boot[
        "event_discount_depth"
    ].clip(
        lower=0.0,
        upper=0.80,
    )

    ordinary_price_log_effect = (
        log_price_coefficient
        * events_boot[
            "current_price_change_residual"
        ]
    )

    promotion_lift_event = (
        events_boot[
            "current_lift_log"
        ]
        - ordinary_price_log_effect
    )

    row = {
        "bootstrap_id": bootstrap_id,
        "panels_sampled": (
            len(common_panels)
        ),
        "events_effective": float(
            event_weights.sum()
        ),
        "mean_discount_depth": (
            weighted_mean(
                depth,
                event_weights,
            )
        ),
        "mean_price_change_residual": (
            weighted_mean(
                events_boot[
                    "current_price_change_residual"
                ],
                event_weights,
            )
        ),
        "current_lift_log": (
            weighted_mean(
                events_boot[
                    "current_lift_log"
                ],
                event_weights,
            )
        ),
        "ordinary_price_lift_log": (
            weighted_mean(
                ordinary_price_log_effect,
                event_weights,
            )
        ),
        "promotion_lift_log": (
            weighted_mean(
                promotion_lift_event,
                event_weights,
            )
        ),
        "current_depth_slope": (
            weighted_slope(
                depth,
                events_boot[
                    "current_lift_log"
                ],
                event_weights,
            )
        ),
        "regular_log_price_coefficient": (
            log_price_coefficient
        ),
        "price_elasticity": elasticity,
    }

    for lag in (
        POST_LAGS_FOR_CALIBRATION
    ):
        dip_column = (
            f"post{lag}_dip_log"
        )

        row[
            f"post{lag}_dip_log"
        ] = weighted_mean(
            events_boot[
                dip_column
            ],
            event_weights,
        )

        row[
            f"post{lag}_depth_slope"
        ] = weighted_slope(
            depth,
            events_boot[
                dip_column
            ],
            event_weights,
        )

    bootstrap_rows.append(row)

bootstrap_moments = pd.DataFrame(
    bootstrap_rows
)

display(
    bootstrap_moments.describe(
        percentiles=[
            0.05,
            0.50,
            0.95,
        ]
    ).T
)

## 9. Map moments into interpretable behavioral parameter draws

The revised dynamic model is

\[
Q_t
=
Q_0(1-d_t)^{-\varepsilon}
\exp\!\left(
    \gamma\mathbf 1\{d_t>0\}
    -\psi I_t
\right),
\]

\[
I_{t+1}=rI_t+d_t.
\]

For an isolated promotion of depth \(d\) starting from \(I_t=0\),

\[
\log Q_t-\log Q_t^{0}
=
-\varepsilon\log(1-d)+\gamma,
\]

while the depth-dependent demand dip one week later is approximately

\[
D_1(d)\approx\psi d.
\]

Subsequent displacement slopes satisfy approximately

\[
D_h(d)\approx\psi r^{h-1}d.
\]

This mapping separates the contemporaneous merchandising effect
\(\gamma\) from dynamic displacement. Persistence \(r\) is estimated
from the decay of depth slopes only when the post-period data contain
a usable signal. Otherwise, the saved behavioral distribution expands
over a prespecified partial-identification set.

In [ ]:
def fit_persistence(
    row: pd.Series,
) -> tuple[float, bool, float]:
    slopes = np.array(
        [
            row.get(
                f"post{lag}_depth_slope",
                np.nan,
            )
            for lag in (
                POST_LAGS_FOR_CALIBRATION
            )
        ],
        dtype=float,
    )

    displacement = float(
        np.clip(
            slopes[0]
            if np.isfinite(slopes[0])
            else 0.0,
            *DISPLACEMENT_BOUNDS,
        )
    )

    second_slope = (
        float(slopes[1])
        if len(slopes) > 1
        and np.isfinite(slopes[1])
        else np.nan
    )

    informative = bool(
        displacement >= MIN_PERSISTENCE_SIGNAL
        and np.isfinite(second_slope)
        and second_slope > 0.0
        and second_slope
        <= (
            MAX_PERSISTENCE_SLOPE_RATIO
            * displacement
        )
    )

    if not informative:
        return (
            float(
                np.median(
                    PERSISTENCE_PARTIAL_ID_GRID
                )
            ),
            False,
            np.nan,
        )

    persistence = float(
        np.clip(
            second_slope
            / displacement,
            *PERSISTENCE_BOUNDS,
        )
    )

    fitted_second_slope = (
        displacement
        * persistence
    )
    fit_loss = float(
        (
            second_slope
            - fitted_second_slope
        )
        ** 2
    )

    return (
        persistence,
        True,
        fit_loss,
    )

base_parameter_draws = (
    bootstrap_moments.copy()
)

fitted_persistence = (
    base_parameter_draws.apply(
        fit_persistence,
        axis=1,
        result_type="expand",
    )
)

fitted_persistence.columns = [
    "inventory_persistence_hat",
    "persistence_data_informative",
    "persistence_fit_loss",
]

base_parameter_draws = pd.concat(
    [
        base_parameter_draws,
        fitted_persistence,
    ],
    axis=1,
)

base_parameter_draws[
    "promotion_lift_log"
] = base_parameter_draws[
    "promotion_lift_log"
].clip(
    *PROMOTION_LIFT_BOUNDS
)

base_parameter_draws[
    "displacement_strength"
] = base_parameter_draws[
    "post1_depth_slope"
].fillna(0.0).clip(
    *DISPLACEMENT_BOUNDS
)

expanded_rows = []

for row in base_parameter_draws.itertuples(
    index=False
):
    row_dict = row._asdict()

    if bool(
        row_dict[
            "persistence_data_informative"
        ]
    ):
        persistence_values = [
            float(
                row_dict[
                    "inventory_persistence_hat"
                ]
            )
        ]
        persistence_source = (
            "geometric_depth_slope_fit"
        )
    else:
        persistence_values = list(
            PERSISTENCE_PARTIAL_ID_GRID
        )
        persistence_source = (
            "partial_identification_grid"
        )

    conditional_weight = (
        1.0
        / len(persistence_values)
    )

    for persistence in (
        persistence_values
    ):
        expanded = dict(row_dict)
        expanded[
            "inventory_persistence"
        ] = float(persistence)
        expanded[
            "persistence_source"
        ] = persistence_source
        expanded[
            "draw_weight"
        ] = (
            conditional_weight
            / BOOTSTRAP_REPLICATIONS
        )
        expanded_rows.append(expanded)

parameter_draws = pd.DataFrame(
    expanded_rows
)

regular_reference = (
    calibration_history.loc[
        calibration_history[
            "promotion_indicator"
        ].eq(0)
        & calibration_history[
            "post_promotion_indicator"
        ].eq(0)
    ]
    .copy()
)

base_demand = float(
    regular_reference[
        "move"
    ].median()
)
regular_price = float(
    regular_reference[
        "regular_price_model"
    ].median()
)
unit_cost = float(
    calibration_history[
        "unit_cost_observed"
    ].median()
)

if (
    not np.isfinite(base_demand)
    or base_demand <= 0
):
    raise ValueError(
        "Invalid calibrated base demand."
    )

if (
    not np.isfinite(regular_price)
    or regular_price <= 0
):
    raise ValueError(
        "Invalid calibrated regular price."
    )

if (
    not np.isfinite(unit_cost)
    or unit_cost <= 0
):
    raise ValueError(
        "Invalid calibrated unit cost."
    )

for frame in [
    base_parameter_draws,
    parameter_draws,
]:
    frame["base_demand"] = (
        base_demand
    )
    frame["regular_price"] = (
        regular_price
    )
    frame["unit_cost"] = (
        unit_cost
    )
    frame["calibration_end_week"] = (
        CALIBRATION_END_WEEK
    )

parameter_columns = [
    "price_elasticity",
    "promotion_lift_log",
    "displacement_strength",
    "inventory_persistence",
]

bounds = {
    "price_elasticity": (
        ELASTICITY_BOUNDS
    ),
    "promotion_lift_log": (
        PROMOTION_LIFT_BOUNDS
    ),
    "displacement_strength": (
        DISPLACEMENT_BOUNDS
    ),
    "inventory_persistence": (
        PERSISTENCE_BOUNDS
    ),
}

boundary_rows = []

for column in parameter_columns:
    lower, upper = bounds[column]
    boundary_rows.append(
        {
            "parameter": column,
            "lower_boundary_share": (
                np.isclose(
                    parameter_draws[
                        column
                    ],
                    lower,
                ).mean()
            ),
            "upper_boundary_share": (
                np.isclose(
                    parameter_draws[
                        column
                    ],
                    upper,
                ).mean()
            ),
        }
    )

clipping_diagnostics = pd.DataFrame(
    boundary_rows
)

print("Base demand:", base_demand)
print("Regular price:", regular_price)
print("Unit cost:", unit_cost)
print(
    "Expanded behavioral draws:",
    len(parameter_draws),
)
print(
    "Persistence empirically informative:",
    f"{base_parameter_draws['persistence_data_informative'].mean():.2%}",
)
print(
    "Total draw weight:",
    f"{parameter_draws['draw_weight'].sum():.6f}",
)

display(clipping_diagnostics)
display(
    parameter_draws[
        parameter_columns
    ].describe(
        percentiles=[
            0.05,
            0.50,
            0.95,
        ]
    ).T
)
display(
    parameter_draws.groupby(
        "persistence_source",
        observed=True,
    )[
        "draw_weight"
    ]
    .sum()
    .rename(
        "probability_mass"
    )
    .reset_index()
)

## 10. Bootstrap pooled and depth-specific event-study uncertainty

In [ ]:
event_plot_data = event_long.loc[
    event_long[
        "event_id"
    ].isin(
        event_summary[
            "event_id"
        ]
    )
].copy()

event_attributes = (
    event_summary[
        [
            "event_id",
            "depth_group",
            "calibration_event",
        ]
    ]
    .drop_duplicates("event_id")
)

event_plot_data = (
    event_plot_data.merge(
        event_attributes,
        on="event_id",
        how="left",
        validate="many_to_one",
    )
)

point_event_study = (
    event_plot_data.groupby(
        "relative_week",
        observed=True,
    )
    .agg(
        mean_effect=(
            "event_effect",
            "mean",
        ),
        observations=(
            "event_effect",
            "count",
        ),
        events=(
            "event_id",
            "nunique",
        ),
    )
    .reset_index()
)

point_depth_event_study = (
    event_plot_data.loc[
        event_plot_data[
            "calibration_event"
        ]
    ]
    .groupby(
        [
            "relative_week",
            "depth_group",
        ],
        observed=True,
    )
    .agg(
        mean_effect=(
            "event_effect",
            "mean",
        ),
        observations=(
            "event_effect",
            "count",
        ),
        events=(
            "event_id",
            "nunique",
        ),
    )
    .reset_index()
)

descriptive_panels = np.array(
    sorted(
        event_plot_data[
            "store_upc"
        ].unique()
    )
)

bootstrap_event_rows = []
bootstrap_depth_rows = []
rng_event = np.random.default_rng(
    RANDOM_SEED + 1
)

for bootstrap_id in range(
    BOOTSTRAP_REPLICATIONS
):
    sampled_panels = rng_event.choice(
        descriptive_panels,
        size=len(descriptive_panels),
        replace=True,
    )

    weights = (
        pd.Series(sampled_panels)
        .value_counts()
        .rename("cluster_weight")
    )

    boot = event_plot_data.merge(
        weights,
        left_on="store_upc",
        right_index=True,
        how="inner",
        validate="many_to_one",
    )

    for relative_week, group in (
        boot.groupby(
            "relative_week",
            observed=True,
        )
    ):
        bootstrap_event_rows.append(
            {
                "bootstrap_id": (
                    bootstrap_id
                ),
                "relative_week": int(
                    relative_week
                ),
                "mean_effect": (
                    weighted_mean(
                        group[
                            "event_effect"
                        ],
                        group[
                            "cluster_weight"
                        ].astype(float),
                    )
                ),
            }
        )

    depth_boot = boot.loc[
        boot[
            "calibration_event"
        ]
    ]

    for (
        relative_week,
        depth_group,
    ), group in depth_boot.groupby(
        [
            "relative_week",
            "depth_group",
        ],
        observed=True,
    ):
        bootstrap_depth_rows.append(
            {
                "bootstrap_id": (
                    bootstrap_id
                ),
                "relative_week": int(
                    relative_week
                ),
                "depth_group": (
                    str(depth_group)
                ),
                "mean_effect": (
                    weighted_mean(
                        group[
                            "event_effect"
                        ],
                        group[
                            "cluster_weight"
                        ].astype(float),
                    )
                ),
            }
        )

bootstrap_event = pd.DataFrame(
    bootstrap_event_rows
)

interval_event_study = (
    bootstrap_event.groupby(
        "relative_week",
        observed=True,
    )[
        "mean_effect"
    ]
    .quantile(
        list(EVENT_STUDY_CI)
    )
    .unstack()
    .rename(
        columns={
            EVENT_STUDY_CI[0]: (
                "ci_lower"
            ),
            EVENT_STUDY_CI[1]: (
                "ci_upper"
            ),
        }
    )
    .reset_index()
)

event_study_table = (
    point_event_study.merge(
        interval_event_study,
        on="relative_week",
        how="left",
        validate="one_to_one",
    )
    .sort_values(
        "relative_week"
    )
)

bootstrap_depth = pd.DataFrame(
    bootstrap_depth_rows
)

interval_depth_event_study = (
    bootstrap_depth.groupby(
        [
            "relative_week",
            "depth_group",
        ],
        observed=True,
    )[
        "mean_effect"
    ]
    .quantile(
        list(EVENT_STUDY_CI)
    )
    .unstack()
    .rename(
        columns={
            EVENT_STUDY_CI[0]: (
                "ci_lower"
            ),
            EVENT_STUDY_CI[1]: (
                "ci_upper"
            ),
        }
    )
    .reset_index()
)

depth_event_study_table = (
    point_depth_event_study.merge(
        interval_depth_event_study,
        on=[
            "relative_week",
            "depth_group",
        ],
        how="left",
        validate="one_to_one",
    )
    .sort_values(
        [
            "depth_group",
            "relative_week",
        ]
    )
)

display(event_study_table)
display(
    depth_event_study_table.loc[
        depth_event_study_table[
            "relative_week"
        ].between(0, 4)
    ]
)

## 11. Empirical figures

In [ ]:
fig, ax = plt.subplots(
    figsize=(8, 4.8)
)

ax.plot(
    event_study_table[
        "relative_week"
    ],
    event_study_table[
        "mean_effect"
    ],
    marker="o",
    label="All isolated promotions",
)
ax.fill_between(
    event_study_table[
        "relative_week"
    ],
    event_study_table[
        "ci_lower"
    ],
    event_study_table[
        "ci_upper"
    ],
    alpha=0.20,
    label=(
        "90% cluster-bootstrap interval"
    ),
)
ax.axhline(
    0.0,
    linewidth=1.0,
)
ax.axvline(
    0.0,
    linestyle="--",
    linewidth=1.0,
)
ax.set_xlabel(
    "Week relative to isolated promotion"
)
ax.set_ylabel(
    "Residual log demand relative to pre-period"
)
ax.set_title(
    "Promotion lift and post-promotion demand path"
)
ax.legend()
fig.tight_layout()
fig.savefig(
    EVENT_STUDY_FIGURE_PATH,
    dpi=300,
    bbox_inches="tight",
)
plt.show()

fig, ax = plt.subplots(
    figsize=(8, 4.8)
)

for depth_group, group in (
    depth_event_study_table.groupby(
        "depth_group",
        observed=True,
    )
):
    group = group.sort_values(
        "relative_week"
    )
    ax.plot(
        group[
            "relative_week"
        ],
        group[
            "mean_effect"
        ],
        marker="o",
        label=str(depth_group),
    )

ax.axhline(
    0.0,
    linewidth=1.0,
)
ax.axvline(
    0.0,
    linestyle="--",
    linewidth=1.0,
)
ax.set_xlabel(
    "Week relative to isolated promotion"
)
ax.set_ylabel(
    "Residual log demand relative to pre-period"
)
ax.set_title(
    "Promotion response by measured discount depth"
)
ax.legend()
fig.tight_layout()
fig.savefig(
    DEPTH_EVENT_STUDY_FIGURE_PATH,
    dpi=300,
    bbox_inches="tight",
)
plt.show()

print(
    "Saved:",
    EVENT_STUDY_FIGURE_PATH,
)
print(
    "Saved:",
    DEPTH_EVENT_STUDY_FIGURE_PATH,
)

## 12. Save behavioral calibration artifacts

In [ ]:
moment_summary = (
    bootstrap_moments.describe(
        percentiles=[
            0.05,
            0.50,
            0.95,
        ]
    )
    .T
    .reset_index()
    .rename(
        columns={
            "index": "moment",
        }
    )
)

parameter_summary = (
    parameter_draws[
        [
            "price_elasticity",
            "promotion_lift_log",
            "displacement_strength",
            "inventory_persistence",
            "base_demand",
            "regular_price",
            "unit_cost",
        ]
    ]
    .describe(
        percentiles=[
            0.05,
            0.50,
            0.95,
        ]
    )
    .T
    .reset_index()
    .rename(
        columns={
            "index": "parameter",
        }
    )
)

parameter_draws.to_pickle(
    PARAMETER_OUTPUT_PATH
)
base_parameter_draws.to_pickle(
    BASE_PARAMETER_OUTPUT_PATH
)
event_study_table.to_csv(
    EVENT_STUDY_TABLE_PATH,
    index=False,
)
depth_event_study_table.to_csv(
    DEPTH_EVENT_STUDY_TABLE_PATH,
    index=False,
)
moment_summary.to_csv(
    MOMENT_TABLE_PATH,
    index=False,
)
parameter_summary.to_csv(
    PARAMETER_SUMMARY_PATH,
    index=False,
)

configuration = {
    "primary_model": PRIMARY_MODEL,
    "calibration_end_week": int(
        CALIBRATION_END_WEEK
    ),
    "evaluation_start_week": int(
        EVALUATION_START_WEEK
    ),
    "event_window": (
        EVENT_WINDOW.tolist()
    ),
    "pre_event_weeks": list(
        PRE_EVENT_WEEKS
    ),
    "post_lags_for_calibration": list(
        POST_LAGS_FOR_CALIBRATION
    ),
    "selected_isolation_gap": list(
        selected_gap
    ),
    "selected_calibration_discount": float(
        selected_calibration_discount
    ),
    "isolated_events": int(
        len(event_summary)
    ),
    "depth_calibration_events": int(
        calibration_event_summary.shape[0]
    ),
    "bootstrap_replications": int(
        BOOTSTRAP_REPLICATIONS
    ),
    "random_seed": int(
        RANDOM_SEED
    ),
    "parameter_mapping": {
        "demand": (
            "Q0*(1-d)^(-epsilon)*"
            "exp(gamma*1[d>0]-psi*I)"
        ),
        "inventory": (
            "I_next=r*I+d"
        ),
        "promotion_lift": (
            "event0_demand_residual_change-beta_price*event0_price_residual_change"
        ),
        "displacement_strength": (
            "slope_of_week1_dip_on_discount_depth"
        ),
        "persistence": (
            "identified_only_when_week1_and_week2_depth_slopes_are_positive_and_decay_consistent;otherwise_partial_identification"
        ),
    },
    "persistence_partial_id_grid": list(
        PERSISTENCE_PARTIAL_ID_GRID
    ),
    "calibration_role": (
        "reduced_form_moment_calibration_"
        "not_causal_structural_identification"
    ),
}

CONFIGURATION_PATH.write_text(
    json.dumps(
        configuration,
        indent=2,
    ),
    encoding="utf-8",
)

print(
    "Saved behavioral draws:",
    PARAMETER_OUTPUT_PATH,
)
print(
    "Saved base bootstrap draws:",
    BASE_PARAMETER_OUTPUT_PATH,
)
print(
    "Saved event-study table:",
    EVENT_STUDY_TABLE_PATH,
)
print(
    "Saved depth event-study table:",
    DEPTH_EVENT_STUDY_TABLE_PATH,
)
print(
    "Saved parameter summary:",
    PARAMETER_SUMMARY_PATH,
)
print(
    "Saved configuration:",
    CONFIGURATION_PATH,
)

## Interpretation checklist

Before using the behavioral distribution in notebook 05:

1. Event demand and price effects must be compared on the same
   fixed-effect-residualized scale.
2. The contemporaneous promotion-lift parameter may be negative; it
   is not truncated to manufacture promotion profitability.
3. The depth-specific analysis should show whether deeper promotions
   are followed by larger week-\(+1\) demand dips.
4. Persistence is empirically informative only when both week-\(+1\)
   and week-\(+2\) depth slopes are positive and consistent with
   geometric decay.
5. Otherwise, persistence is represented by the transparent
   partial-identification sensitivity grid.
6. Promotions below the measured-discount threshold remain in the
   descriptive event study but do not identify the dynamic mechanism.
7. Point estimates and bootstrap intervals use the same descriptive
   panel population.
8. Large boundary shares or near-zero displacement imply that the
   stockpiling channel is empirically weak.
9. All estimates are calibration moments, not causal structural
   parameters.